# Quaternion

REF: Using math expressions w/ LaTeX
- **Inline Math** (appears inside a sentence): Wrap your LaTeX code in a single dollar sign `$`
   - $E = mc^2$

- **Display / Block Math** (appears centered on its own line): Wrap your LaTeX code in a double dollar sign `$$`
   - $$\int_{a}^{b} f(x) \, dx$$

***Common LaTeX Syntax Quick REF***
- Subscripts & Superscripts: `x_i^2` creates $x_i^2$
- Fractions: `\frac{a}{b}` creates $\frac{a}{b}$
- Square Roots: `\sqrt{x}`  or `\sqrt[n]{x}` creates $\sqrt{x}$ and $\sqrt[n]{x}$
- Greek Letters: `\alpha`, `\beta`, `\gamma`, `\pi` gives $\alpha$, $\beta$, $\gamma$, $\pi$
- Summations: `\sum_{i=1}^{n} i` gives $\sum_{i=1}^{n} i$

>REF: <kalman_bayesian_filter.ipynb>

## 1. What is **quaternion**?
In **MuJoCo**, it represents a 3D rotation using four numbers like such:
$$ q = [w, x, y, z]$$

In **ROS** & **SciPy**, instead, it is represented as such:
$$ q = [x, y, z, w]$$

Where,
- `w`: scalar component
- `x, y, z`: vector components (tied to rotation)

A valid quarternion with its components should have unit length:
$$ ||q|| = \sqrt{w^2 + x^2 + y^2 + z^2} = 1 $$


Generally, `<quat>` is subjected to ***normalization*** such that its four numbers scale together to have their combined length to be exactly `1`.

A unit quaternion can represent a rotation by an angle $\theta$ around a unit axis $\hat{u} = (u_x, u_y, u_z)$:
$$ q = \begin{bmatrix} w \\ x \\ y \\ z \end{bmatrix} = \begin{bmatrix} cos(\theta/2) \\ u_x sin(\theta/2) \\ u_y sin(\theta/2) \\ u_z sin(\theta/2) \end{bmatrix}$$


## 2. Normalization
Mentioned above, `<quat>` should have its combined length be `1`. \
In other words, `quaternion` norm should be `1`.

Should a quaternion with norm different from `1`, it is not ***NOT*** a *unit* quaternion. It needs to be normalized first before becoming suitable for operations that expect a unit quaternion.

For a nonzero quaternion `q`, normalization is:
$$q_{normalized} = \frac{q}{||q||}$$
Where,
$$ ||q||= \sqrt{w^2 + x^2 + y^2 + z^2} $$

Example: \
If `q = [2,0,0,0]`, the norm is `2`. Dividing every component by `2` gives `[1,0,0,0]`.

---

# Using Python to compute

## 1. Manually

In [2]:
import numpy as np
import math
q = np.array([2,0,0,0], dtype=float)
print('OG q: ',q)
norm = math.sqrt(q[0]**2 + q[1]**2 + q[2]**2 + q[3]**2)
print("OG norm: ", norm)

q_normalized = q / norm
print('\nNormalized q: ', q_normalized)
q_normalized_norm = math.sqrt(q_normalized[0]**2 + q_normalized[1]**2 + q_normalized[2]**2 + q_normalized[3]**2)
print('Normalized norm: ', q_normalized_norm)

OG q:  [2. 0. 0. 0.]
OG norm:  2.0

Normalized q:  [1. 0. 0. 0.]
Normalized norm:  1.0


## 2. `np.linalg.norm()`
`np.linalg.norm()` calculates the **length** or **magnitude** of a vector for us.

It does the following computation:
$$ ||q||= \sqrt{w^2 + x^2 + y^2 + z^2} $$

For a quaternion:

In [3]:
q = np.array([2.0, 0.0, 0.0, 0.0])
norm = np.linalg.norm(q)
print(norm) # expects 2

2.0


Embedded into a function:

In [4]:
# computes the length of the vector
def quaternion_norm(q):
    q = np.asarray(q, dtype=float)
    return np.linalg.norm(q)

q = np.array([2.0, 0.0, 0.0, 0.0])

print("OG quaternion:", q)
print("OG length:", quaternion_norm(q))

q_normalized = q / quaternion_norm(q)

print("\nNormalized quaternion:", q_normalized)
print("Normalized length:", quaternion_norm(q_normalized))
print(q_normalized.shape)

OG quaternion: [2. 0. 0. 0.]
OG length: 2.0

Normalized quaternion: [1. 0. 0. 0.]
Normalized length: 1.0
(4,)


### Add-Ons:

`np.linalg.norm()` can take the following arguments:\
`np.linalg.norm(x, ord=None, axis=None, keepdims=False)`

If there's a need to normalize multiple quaternions at once, we can alter the following args:
- `axis` specifies the axis (or axes for matrices) along which to compute the norm
- `keepdims` if set to `True`, the axes that are normed are left in the result as dimensions with size one

#### `axis` parameter
By default, `axis=None`. \
It flattens your entire array and calculates a single number.\
Specifying an axis lets you collapse only certain dimensions.

**For a 2D Array (Matrix) - Having multiple quaternions**\
Matrix has **rows** (`axis=0`, hori. direction) and **columns** (`axis=1`, vert. direction).
- `axis=0` **(Along Columns):** Collapses the rows. It calculates the norm of each column independently.
- `axis=1` **(Along Rows):** Collapses the columns. It calculates the norm of each row independently.

In [5]:
x = np.array([[3,4],
              [6,8]])
y = np.array([2,0,0,0])
z = np.array([[2,0,0,0]])

print('x:', x.shape, x.ndim)
print('y:', y.shape, y.ndim)
print('z:', z.shape, z.ndim)

x: (2, 2) 2
y: (4,) 1
z: (1, 4) 2


In [6]:
x = np.array([[3,4],
              [6,8]])

# axis=0: Norm of each column vector: [sqrt(3²+6²), sqrt(4²+8²)]
print('axis = 0, ',np.linalg.norm(x, axis=0))

# axis=1: Norm of each row vector: [sqrt(3²+4²), sqrt(6²+8²)]
print('axis = 1, ',np.linalg.norm(x, axis=1))

axis = 0,  [6.70820393 8.94427191]
axis = 1,  [ 5. 10.]


- `axis=-1` refers to the **very last axis** of the presented array.\
For a 2D-array, `axis=1` and `axis=-1` both collapses the columns and calculate the norm of the row.

In [7]:
x = np.array([[3,4],
              [6,8]])

# axis=-1: Norm of each row vector: [sqrt(3²+4²), sqrt(6²+8²)]
print('axis = -1, ',np.linalg.norm(x, axis=-1))

axis = -1,  [ 5. 10.]


#### `keepdims` parameter
When a norm is calculated along an axis, `Numpy` removes that collapsed axis entirely because it has been collapsed into a single number;\
meaning you just lost a dimension.

By setting `keepdims = True`, `Numpy` **keeps** that collapsed axis as a ***dimension of size 1***.

In [8]:
x = np.array([[3, 4],
              [6, 8]])

# Without keepdims: result drops down to 1D
norm_false = np.linalg.norm(x, axis=1, keepdims=False)
print('norm_false: ', norm_false)
print('norm_false shape: ', norm_false.shape)  
# Output: (2,)  -> Vector
print('\n')
# With keepdims: result stays 2D, with column size 1; (row,column) --> (2,1)
norm_true = np.linalg.norm(x, axis=1, keepdims=True)
print('norm_true:\n', norm_true)
print('norm_true shape: ',norm_true.shape)  
# Output: (2, 1) -> Column Matrix


norm_false:  [ 5. 10.]
norm_false shape:  (2,)


norm_true:
 [[ 5.]
 [10.]]
norm_true shape:  (2, 1)


### Combined `axis` and `keepdims` example
Handling a batch of quaternions, subjected to normalization.


In [9]:
# say we have a list of 3 diff quat 
q = np.array([[2,0,0,0],
              [1,0,0,1],
              [0,0,0,1]])
print('q.shape: ',q.shape)

# calc individual length|norm
length = np.linalg.norm(q, axis=-1,keepdims=True)
print('length:\n',length)
print(length.shape)
print(length.ndim)

# normalize each <quat> individual component
q_normalized = q/length
print('q_normalized:\n', q_normalized)
print('q_normalized (dims):\n', q_normalized.ndim)

q.shape:  (3, 4)
length:
 [[2.        ]
 [1.41421356]
 [1.        ]]
(3, 1)
2
q_normalized:
 [[1.         0.         0.         0.        ]
 [0.70710678 0.         0.         0.70710678]
 [0.         0.         0.         1.        ]]
q_normalized (dims):
 2


In [10]:
# say we have a list of 3 diff quat 
q = np.array([[2,0,0,0],
              [1,0,0,1],
              [0,0,0,1]])
print('q.shape: ',q.shape)

# calc individual length|norm, if we set keepdims=False, we can expect error later
length = np.linalg.norm(q, axis=-1,keepdims=False)
print('length:\n',length)
print(length.shape)
print(length.ndim)

# normalize each <quat> individual component
q_normalized = q/length
print('q_normalized:\n', q_normalized)
print('q_normalized (dims):\n', q_normalized.ndim)

q.shape:  (3, 4)
length:
 [2.         1.41421356 1.        ]
(3,)
1


ValueError: operands could not be broadcast together with shapes (3,4) (3,) 

### In a function

In [ ]:
def quat_normalize(q):
    length = np.linalg.norm(q, axis=-1, keepdims=1)
    q_normalized = q / length
    return q_normalized

q = np.array([[2,0,0,0],
              [1,0,0,1],
              [0,0,0,1]])

normalized_batch = quat_normalize(q)
print(normalized_batch)
print(normalized_batch[0])
print(normalized_batch[1])
print(normalized_batch[2])

[[1.         0.         0.         0.        ]
 [0.70710678 0.         0.         0.70710678]
 [0.         0.         0.         1.        ]]
[1. 0. 0. 0.]
[0.70710678 0.         0.         0.70710678]
[0. 0. 0. 1.]


### A quick ref of quaternion to euler
| Quaternion (w,x,y,z) | Rotation |
|---|---|
| `np.array([1,0,0,0])` | none (0°)|
| `np.array([1,0,0,1])` | 90° (z-axis) |
| `np.array([0,0,0,1])` | 180° (z-axis) |
| `np.array([1,1,0,0])` | 90° (x-axis) |

---

### Further Review (Safety Check)
There's a possible breakpoint identified, doesn't cause an Error, but will affect the computation, math and result.

`<quat>` should have **4** components presented as `[X,X,X,X]`, either initially being presented as `list` or `np.array`:
| Code | Type | Shape |
|---|---|---|
| `[0,0,0,1]` | `List` |  `List` has no attribute `shape` | 
| `np.array([0,0,0,1])` | `Numpy Array` | (4, ) |

Breakpoint is when `<quat>` is input wrongly by user, such as:
```python
quat = [0,0,1] # we're missing one more value, system can't flag this, 
               # but by numpy std, this is only shape (3, )
```
unlike:
```python
q = np.array([[2,0,0,0],
              [1,0,0,1],
              [0,0,0,-1],
              [1,0,1,0]])
```
where mismatched arrays will be flagged by the system with `ValueError` as shown:

In [ ]:
q = np.array([[2,0,0,0],
              [1,0,0,1],
              [0,0,0,-1],
              [1,0,1]]) # last row is missing one value

ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (4,) + inhomogeneous part.

Obj is to flag any cases where `<quat>` only contains 3 components, to do that:\
Integrate a ***safety check*** where any form of `<quat>` is: \
**(1) converted to a `numpy array`** 
>`x = np.asarray(x)`

and

**(2) have 4 components** prior to any math computation, which could be presented in the following form:
```python
q = [2,0,0,0]
q = np.asarray(q)  # shape (4, )

z = np.array([[2, 0, 0, 0]])  # shape (1, 4)
```
Taking into consideration of both forms, we'll use `assert` command like such:
>`assert q.shape[-1] == 4` ; this will target key tgt area that would contain the '4'

If the value isn't `4`, then `AssertionError` triggers:

In [12]:
q = [0,0,1]
q = np.asarray(q)
assert q.shape[-1] == 4

AssertionError: 

## Finalized Code
For `<robot_data.py>`:

In [67]:
def quat_normalize(q):
    q = np.asarray(q) # 'convert' to np.array
    assert q.shape[-1] == 4 # check if there's 4 components

    norm = np.linalg.norm(q, axis = -1, keepdims=True)
    q_normalized = q / norm
    return q_normalized

q = [1,0,0,1]
post_q = quat_normalize(q)
print(post_q)
q0, q1, q2, q3 = post_q
print(q0, q1, q2, q3)

q0 = post_q[0]
print(q0)

[0.70710678 0.         0.         0.70710678]
0.7071067811865475 0.0 0.0 0.7071067811865475
0.7071067811865475


**KEEP IN MIND**
| Before-Class | Process | After-Class
|---|---|---|
| <class `list`> | `np.linalg.norm()` |  <class `numpy.ndarray`> |
| <class `numpy.ndarray` or `numpy.array`> | `np.linalg.norm()` | <class `numpy.ndarray`> |

> No issue if input is a `<list>`, there won't be any math error at all, but the thing is we can't fact-check the shape with a list, thus we need to push it to become a numpy array to fact-check the shape (4, ) first before doing the math on it.

---

# `<class Robot>` Constructor(s)
Notes on the following lines of code:
```python
def __init__(self):
    self.body = {}
    self.misc = None
```

Recap: \
`__init__` runs automatically when this line is proc: 
```python
# call the class; create an <object> from this <class>
robot = Robot()
```
It sets up the robot's starting state. Think of it as building a filing cabinet. \
`__init__` installs the following compartments:
1) a drawer labelled `body` (empty, but ready to hold - folders in this case since we have many `add_body()`)
2) a tray labelled  `misc` (empty, but ready to hold - just one doc since we only have one `add_misc()`)

Without the constructor containing `self.body` or `self.misc`, calling `add_body()` or `add_misc()` results in:
```python
AttributeError:'Robot' object has no attribute 'body'
AttributeError: 'Robot' object has no attribute 'misc'
```

`self` means "the robot being built", so `self.body = {}` attaches a storage spot called `body` to that robot.

## Storage space for `<class Robot>`
Reasoning for why `self.body = {}` and `self.misc = None`.

These are chosen based on how many things each one will hold:
|  | To hold | Start as | Reason
|---|---|---|---|
| `self.body` | six bodies, added one by one |  `{}`, empty dict | Empty container needs to exist first before being able to add things to it |
| `self.misc` | one object | `None`, "nothing here yet" | A single item has nothing to be a container for |


## Pertaining to `body`
### Using `dictionary {}` instead of `list []`
Using `dict` is easier to store things under a key of my choosing, in my case, `key` is `body_id` and it runs from body ids 1 thru 6.\
In `add_body()`, I set it up as:
```python
self.body[body_id] = Robot.Body(parent, name, pos, quat, mass, ipos, iquat, inertia, joint_axis, joint_range)
```
`dict` also helps with keeping things in the order I added them, looping with `.items()` visits bodies 1 through 6 in order.

While a `list` would work, it just isn't as flexible as a `dict`.

---

# Determining Axis of Rotation | Rotation Axis
There's need to discern the rotation axis (x,y,z) from the following inputs taken from `<ur5e.xml>` and recorded in `<robot_data.py>`:
```python
joint_axis = [1,0,0] # rotates about x-axis
           = [0,1,0] # rotates about y-axis
           = [0,0,1] # rotates about z-axis
```


## 1) Manual

In [42]:
joint_axis = [0,0,1]

index_holder = 0
for index, value in enumerate(joint_axis):
    print(index, value)
    if value > index_holder:
        index_holder = index

print('index_holder: ',index_holder)
    

0 0
1 0
2 1
index_holder:  2


## 2) `np.argmax()`
We can use `np.argmax()` to return the ***indices of the maximum values*** along a specified axis in an array. It'll tell you *exactly where* that largest number is located.

`np.argmax()` takes in the following arg:

```python
np.argmax(x, axis=None, keepdims=False)
```

If no axis is provided, cmd flattens the array and gives you a single index.

In [52]:
x = np.array([10, 42, 5, 42, 30])

# Find the index of the maximum value
index = np.argmax(x) 

print(index) 
# Output: 1 (Even tho 42 appears twice, it picks the first one at index 1)

1


### Multi-dimensional Arrays (`axis=0` vs `axis=1`)
When dealing with matrices, we can find the maximum positions across columns or rows.

In [53]:
matrix = np.array([[15, 20, 5],
                   [8,  35, 50],
                   [22, 12, 19]])

# axis=0: Looks down columns to find the row index of the max value
print(np.argmax(matrix, axis=0))
# Output: [2, 1, 1] 
# (Column 1 max is 22 at row 2; Column 2 max is 35 at row 1; Column 3 max is 50 at row 1)

# axis=1: Looks across rows to find the column index of the max value
print(np.argmax(matrix, axis=1))
# Output: [1, 2, 0]
# (Row 1 max is 20 at col 1; Row 2 max is 50 at col 2; Row 3 max is 22 at col 0)

[2 1 1]
[1 2 0]


## Our Solution

In [54]:
q = [0,1,0]
print(q)
print(type(q))
check = np.abs(q)
print(check)
print(type(check))
stance = np.argmax(check) # find the index of the max value
print(stance)

# collapsed into one line
q = [0,0,1]
axis_selection = np.argmax(q)
print(axis_selection)
print(type(axis_selection))

# translate to x, y or z
if axis_selection == 0:
    print('Rotation axis is', 'x')
elif axis_selection == 1:
    print('Rotation axis is','y')
else:
    print('Rotation axis is','z')

[0, 1, 0]
<class 'list'>
[0 1 0]
<class 'numpy.ndarray'>
1
2
<class 'numpy.int64'>
Rotation axis is z


## Quick Comparison w/ other similar functions
| Function | What it returns | Example Output for [10,42,5]
|---|---|---|
|`np.max()`|The actual **highest value**|`42`|
|`np.min()`|The actual **lowest value**|`5`|
|`np.argmax()`|The **index** of the highest value|`1`|
|`np.argmin()`|The **index** of the lowest value|`2`|

---

# Determining Rotation Matrix
We have discerned the axis of rotation, this information helps us with identifying the rotation matrix involved:
```python
x_axis = np.array([[1,      0,          0     ],
                   [0, cos(angle), -sin(angle)],
                   [0, sin(angle), cos(angle) ]])

y_axis = np.array([[cos(angle),   0,  sin(angle)],
                   [    0,        1,      0     ],
                   [-sin(angle),  0, cos(angle) ]])

z_axis = np.array([[cos(angle),  -sin(angle),  0],
                   [sin(angle),   cos(angle),  0],
                   [   0,         0,          1 ]])                   
```

In [60]:
def axis_rotation(rotation_axis_id, angle):
    # assign your cos and sin values for easier callback; angle to be in rad
    c = np.cos(angle)
    s = np.sin(angle)

    # rotation about the x_axis; rotation_axis_id = 0
    if rotation_axis_id == 0:
        rotation_matrix = np.array([[1,0,0],
                                    [0,c,-s],
                                    [0,s,c]])

    # rotation about the y_axis; rotation_axis_id = 1
    elif rotation_axis_id == 1:
        rotation_matrix = np.array([[c,0,s],
                                    [0,1,0],
                                    [-s,0,c]])

    # rotation about the z_axis; rotation_axis_id = 2
    elif rotation_axis_id == 2:
        rotation_matrix = np.array([[c,-s,0],
                                    [s,c,0],
                                    [0,0,1]])

    # we conduct check if rotation_axis_id isn't 0, 1 or 2;
    else:
        raise ValueError("rotation_axis_id must only be 0(x), 1(y) or 2(z). No other values accepted.")

    return rotation_matrix

print(axis_rotation(axis_selection,30))

[[ 0.15425145  0.98803162  0.        ]
 [-0.98803162  0.15425145  0.        ]
 [ 0.          0.          1.        ]]


---

# Factoring in `<quat>` along with the Rotation Matrix computed above
Recap on `quaternion` above, start of this notebook.

Main concept of what we're doing here is trying to describe how `<quat>` is used to rotate the axis frame (x,y,z) from one coordinate frame to another. And then pairing that along with the `<rotation_matrix>` mentioned earlier.
> Think of it as rotating a vector.

For example, suppose a vector points along the (+ve) x-axis :
$$ v = \begin{bmatrix} 1 \\ 0 \\ 0 \end{bmatrix}$$

And now we're looking to have that point along the (+ve) y-axis, like such:
$$ v' = \begin{bmatrix} 0 \\ 1 \\ 0 \end{bmatrix}$$

To do that, we need to rotate $90^\circ$ CCW around the z-axis right? 

That's what the rotation matrix is for, provides a systematic way to perform this calculation as we well know.

To get $v'$, multiply the rotation matrix by $v$:
$$v' = Rv$$

A $90^\circ$ rotation around the z-axis, as shown above, can be represented by:
$$ R_z(90^\circ) = \begin{bmatrix} 0&-1&0 \\ 1&0&0 \\ 0&0&1 \end{bmatrix} \text{   ,where }cos(90^\circ) = 0 \text{  and } sin(90^\circ) = 1 $$

We sub this matrix into the eqn:
$$ v' = R_z(90^\circ)v \\
= \begin{bmatrix} 0&-1&0 \\ 1&0&0 \\ 0&0&1 \end{bmatrix} \begin{bmatrix} 1\\0\\0\end{bmatrix} \\
= \begin{bmatrix} 0\\1\\0\end{bmatrix}$$

## How quaternion 'replaces' rotation matrix in rotating a vector
`<quat>` seeks to do the same, just that unlike rotation matrix which uses 9 numbers, quaternion only uses 4. 

I've already covered briefly about `quaternions` normalization above, this is a continuation from that.

As we're well familiar with `<quat>` by now:
$$ q = \begin{bmatrix} w \\ x \\ y \\ z \end{bmatrix}$$

Where `w` is **scalar**, and `x, y, z` are the **vectors**.

Recall we mentioned earlier that a rotation matrix transforms vectors:
$$ v' = Rv $$
A `quaternion` can perform the same rotation using **quaternion multiplication**.

Say, we represent a 3D vector $ v = (v_x, v_y, v_z)$ as a pure quaternion:
$$ p = \begin{bmatrix} 0 \\ v_x \\ v_y \\ v_z \end{bmatrix}$$

Recall, a unit quaternion can represent a rotation by an angle $\theta$ around a unit axis $\hat{u} = (u_x, u_y, u_z)$:
$$ q = \begin{bmatrix} w \\ x \\ y \\ z \end{bmatrix} = \begin{bmatrix} cos(\theta/2) \\ u_x sin(\theta/2) \\ u_y sin(\theta/2) \\ u_z sin(\theta/2) \end{bmatrix}$$

For a unit quaternion $q$, rotate this vector using:
$$ p' = qq^{-1}p$$
Where,
- $p'$ represents the rotated vector
- $q$ represents the rotation
- $q^{-1}$ is the inverse quaternion
- $p$ represents the og vector

The **inverse** of a **unit quaternion**, is its **conjuate** *(opposite sign)*:
$$ q^{-1} = q^* = \begin{bmatrix} q_0 \\ -q_1 \\ -q_2 \\ -q_3 \end{bmatrix}$$

You may expect the result of $qq^{-1}p$ to have a zero scalar component, and its remaining three components give the rotated vector.

In expanding that multiplication and collecting the coefficients of $v_x,v_y,v_z$, we arrive at the formula that we've been seeking for:
$$ R = \begin{bmatrix} q_0{^2} + q_1{^2} - q_2{^2} - q_3{^2} & 2(q_1 q_2 - q_0 q_3) & 2(q_1 q_3 + q_0 q_2) \\ 2(q_1 q_2 + q_0 q_3) & q_0{^2} - q_1{^2} + q_2{^2} - q_3{^2} & 2(q_2 q_3 - q_0 q_1)
\\ 2(q_1 q_3 - q_0 q_2) & 2(q_2 q_3 + q_0 q_1) & q_0{^2} - q_1{^2} - q_2{^2} + q_3{^2} \end{bmatrix}$$

The matrix presented before you is simply a way to write the quaternion rotation expression as ordinary matrix multiplication.

## Quaternion to Rotation

With all that we've known, we can define a function that takes in `<quat>` and spits out a quaternion rotation representative:

In [71]:
def quat_to_rot(quat):
    # grab individual components from <quat>; these will be our q0, q1, q2, q3
    q0 = quat[0]
    q1 = quat[1]
    q2 = quat[2]
    q3 = quat[3]

    # compute the quaternion rotation
    quat_rot_mat = np.array([[q0**2 + q1**2 - q2**2 - q3**2, 2 * (q1 * q2 - q0 * q3), 2 * (q1 * q3 + q0 * q2)],
                                 [2 * (q1 * q2 + q0 * q3), q0**2 - q1**2 + q2**2 - q3**2, 2 * (q2 * q3 - q0 * q1)],
                                 [2 * (q1 * q3 - q0 * q2), 2 * (q2 * q3 + q0 * q1), q0**2 - q1**2 - q2**2 + q3**2]])
    
    return quat_rot_mat

quat = [1,0,0,1]
post_quat = quat_normalize(quat)
print(post_quat)

print(quat_to_rot(post_quat))

[0.70710678 0.         0.         0.70710678]
[[ 0. -1.  0.]
 [ 1.  0.  0.]
 [ 0.  0.  1.]]


## Combining **quaternion rotation** and **rotation matrix**
Do not be confused between **(1) quaternion rotation** and **(2) rotation matrix**, they both serve different purposes.

To re-iterate:
1) quaternion (refers to both quat and quat_rotation) tracks the global link orientations
2) rotation matrix (axis-angles) models the local joint updates

|  | What is its usage in Forward Kinematics | Why use this? |
|---|---|---|
|**Quaternion**|Tracks the ***global accumulated orientation*** from the base to the end-effector | In computer math, multiplying too many big matrices together leads to messy numbers and could potentially lead the robot to be confused. And so we turn to quat to help prevents errors/drift; takes less memory |
|**Rotation Matrix**| Applying the ***local movement*** around a specific joint's physical axis | Compatible with 3D coords (x,y,z) and translation matrices; grants us the Homogeneous Transforms|


In my own simple terms:\
Because each body has slightly different axis-frames (x,y,z) of different orientations, `quaternion` takes that into account in a math format, and then we also need to take into account if the joint rotates, that's where the `rotation matrix` comes into play.

And so we have 2 'different' rotation matrices on our hands:

In [76]:
print('Quaternion Rotation:\n', quat_to_rot(post_quat))
print('\nRotation Matrix (Axis-Angles): \n', axis_rotation(axis_selection,30))

Quaternion Rotation:
 [[ 0. -1.  0.]
 [ 1.  0.  0.]
 [ 0.  0.  1.]]

Rotation Matrix (Axis-Angles): 
 [[ 0.15425145  0.98803162  0.        ]
 [-0.98803162  0.15425145  0.        ]
 [ 0.          0.          1.        ]]


Both are visibly `3x3` matrices, to combine them both together, adhere to this sequence:
>`Quaternion Rotation @ Rotation Matrix`

`A @ B` and `B @ A` results in 2 very different outcome.

Perhaps you may be curious and wonder why `Quaternion Rotation @ Rotation Matrix`?\
To best answer this, in layman's term, ***apply global effect first, then local effect***. (Refer to the table above if you're lost)